In [1]:

# First Order Logic Resolution
# Prove: John likes peanuts

def is_variable(term):
    return term in {"x", "y", "z", "w", "g", "k"}

def walk(term, substitution):
    while is_variable(term) and term in substitution:
        term = substitution[term]
    return term

def unify_terms(a, b, substitution):
    a = walk(a, substitution)
    b = walk(b, substitution)

    if a == b:
        return True

    if is_variable(a):
        substitution[a] = b
        return True

    if is_variable(b):
        substitution[b] = a
        return True

    return False

def apply_substitution(literal, substitution):
    neg, pred, args = literal

    new_args = tuple(
        walk(term, substitution) for term in args
    )

    return (neg, pred, new_args)

def resolve(c1, i, c2, j):
    l1 = c1[i]
    l2 = c2[j]

    # Check complementary literals
    if l1[0] == l2[0] or l1[1] != l2[1]:
        raise ValueError("Not complementary")

    substitution = {}

    for a, b in zip(l1[2], l2[2]):
        if not unify_terms(a, b, substitution):
            raise ValueError("Unification failed")

    # Remove resolved literals
    result = [
        apply_substitution(l, substitution)
        for k, l in enumerate(c1) if k != i
    ]

    result += [
        apply_substitution(l, substitution)
        for k, l in enumerate(c2) if k != j
    ]

    return result, substitution

def format_literal(literal):
    neg, pred, args = literal
    sign = "NOT " if neg else ""
    return sign + pred + "(" + ", ".join(args) + ")"

def format_clause(clause):
    if not clause:
        return "{}"

    return " OR ".join(
        format_literal(lit) for lit in clause
    )

# Knowledge Base in CNF

C1 = [(True, "Food", ("x",)),
      (False, "Likes", ("John", "x"))]

C2 = [(False, "Food", ("Apple",))]

C3 = [(False, "Food", ("Vegetables",))]

C4 = [(True, "Eats", ("x", "y")),
      (False, "Killed", ("x",)),
      (False, "Food", ("y",))]

C5 = [(False, "Eats", ("Anil", "Peanuts"))]

C6 = [(False, "Alive", ("Anil",))]

C7 = [(True, "Eats", ("Anil", "w")),
      (False, "Eats", ("Harry", "w"))]

C8 = [(False, "Killed", ("g",)),
      (False, "Alive", ("g",))]

C9 = [(True, "Alive", ("k",)),
      (True, "Killed", ("k",))]

# Negation of the query
C10 = [(True, "Likes", ("John", "Peanuts"))]

print("KNOWLEDGE BASE")
for i, clause in enumerate(
    [C1, C2, C3, C4, C5, C6, C7, C8, C9, C10], 1
):
    print(f"C{i}: {format_clause(clause)}")

print("\nPROOF BY RESOLUTION")

# Step 1
R1, s1 = resolve(C1, 1, C10, 0)
print("\nStep 1: C1 and C10")
print("R1:", format_clause(R1))
print("Substitution:", s1)

# Step 2
R2, s2 = resolve(C4, 2, R1, 0)
print("\nStep 2: C4 and R1")
print("R2:", format_clause(R2))
print("Substitution:", s2)

# Step 3
R3, s3 = resolve(R2, 0, C5, 0)
print("\nStep 3: R2 and C5")
print("R3:", format_clause(R3))
print("Substitution:", s3)

# Step 4
R4, s4 = resolve(C9, 1, R3, 0)
print("\nStep 4: C9 and R3")
print("R4:", format_clause(R4))
print("Substitution:", s4)

# Step 5
R5, s5 = resolve(R4, 0, C6, 0)
print("\nStep 5: R4 and C6")
print("R5:", format_clause(R5))

# Check for empty clause
if len(R5) == 0:
    print("\nEmpty clause obtained!")
    print("CONCLUSION: John likes peanuts.")
    print("Hence proved by Resolution.")
else:
    print("\nProof could not be completed.")


KNOWLEDGE BASE
C1: NOT Food(x) OR Likes(John, x)
C2: Food(Apple)
C3: Food(Vegetables)
C4: NOT Eats(x, y) OR Killed(x) OR Food(y)
C5: Eats(Anil, Peanuts)
C6: Alive(Anil)
C7: NOT Eats(Anil, w) OR Eats(Harry, w)
C8: Killed(g) OR Alive(g)
C9: NOT Alive(k) OR NOT Killed(k)
C10: NOT Likes(John, Peanuts)

PROOF BY RESOLUTION

Step 1: C1 and C10
R1: NOT Food(Peanuts)
Substitution: {'x': 'Peanuts'}

Step 2: C4 and R1
R2: NOT Eats(x, Peanuts) OR Killed(x)
Substitution: {'y': 'Peanuts'}

Step 3: R2 and C5
R3: Killed(Anil)
Substitution: {'x': 'Anil'}

Step 4: C9 and R3
R4: NOT Alive(Anil)
Substitution: {'k': 'Anil'}

Step 5: R4 and C6
R5: {}

Empty clause obtained!
CONCLUSION: John likes peanuts.
Hence proved by Resolution.
